In [6]:
file_path1 = os.path.expanduser("~/Downloads/Lung_atlas_public.h5ad")
adata_lung = sc.read_h5ad(file_path1)
adata_lung

AnnData object with n_obs × n_vars = 32472 × 15148
    obs: 'dataset', 'location', 'nGene', 'nUMI', 'patientGroup', 'percent.mito', 'protocol', 'sanger_type', 'size_factors', 'sampling_method', 'batch', 'cell_type', 'donor'
    layers: 'counts'

In [1]:
import os
import numpy as np
import scanpy as sc

file_path = os.path.expanduser("~/Desktop/adata_lung_qc.h5ad")
adata_lung_qc = sc.read_h5ad(file_path)

Loaded baseline object:
AnnData object with n_obs × n_vars = 24824 × 15139
    obs: 'dataset', 'location', 'nGene', 'nUMI', 'patientGroup', 'percent.mito', 'protocol', 'sanger_type', 'size_factors', 'sampling_method', 'batch', 'cell_type', 'donor'
    var: 'n_cells'
    layers: 'counts'

Batch counts:
batch
4     3679
6     3150
A6    2393
5     2378
A5    2335
2     1988
A1    1794
A2    1372
A3    1106
B2     963
1      904
B4     676
A4     667
B1     601
3      445
B3     373
Name: count, dtype: int64


In [2]:
def stratified_subsample_adata(adata, group_key="batch", frac=0.75, random_state=0):
    rng = np.random.default_rng(random_state)
    selected_idx = []

    obs = adata.obs.copy()

    for group, idx in obs.groupby(group_key).indices.items():
        idx = np.array(list(idx))
        n_group = len(idx)
        n_take = max(1, int(np.floor(n_group * frac)))
        chosen = rng.choice(idx, size=n_take, replace=False)
        selected_idx.extend(chosen.tolist())

    selected_idx = np.array(selected_idx)
    return adata[selected_idx].copy()

In [3]:
adata_lung_75_rep1 = stratified_subsample_adata(
    adata_lung_qc,
    group_key="batch",
    frac=0.75,
    random_state=1001
)

adata_lung_75_rep2 = stratified_subsample_adata(
    adata_lung_qc,
    group_key="batch",
    frac=0.75,
    random_state=1002
)

adata_lung_75_rep3 = stratified_subsample_adata(
    adata_lung_qc,
    group_key="batch",
    frac=0.75,
    random_state=1003
)


/var/folders/3w/yklb0sxs0ss499k2l0cb08z80000gn/T/ipykernel_79400/1851489268.py:10: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  for group, idx in obs.groupby(group_key).indices.items():


In [5]:
adata_lung_75_rep1.write(os.path.expanduser("~/Desktop/adata_lung_75_rep1.h5ad"))
adata_lung_75_rep2.write(os.path.expanduser("~/Desktop/adata_lung_75_rep2.h5ad"))
adata_lung_75_rep3.write(os.path.expanduser("~/Desktop/adata_lung_75_rep3.h5ad"))

In [6]:
import os
import gc
import numpy as np
import pandas as pd
import scanpy as sc
import harmonypy as hm
import scib_metrics as sm
from scib_metrics.nearest_neighbors import pynndescent
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score

results_harmony_75 = []

for rep in [1, 2, 3]:
    file_path = os.path.expanduser(f"~/Desktop/adata_lung_75_rep{rep}.h5ad")
    adata_sub = sc.read_h5ad(file_path)
    adata_harmony = adata_sub.copy()
    adata_harmony.X = adata_harmony.layers["counts"].copy()

    sc.pp.normalize_total(adata_harmony, target_sum=1e4)
    sc.pp.log1p(adata_harmony)

    sc.pp.highly_variable_genes(
        adata_harmony,
        flavor="seurat",
        batch_key="batch",
        n_top_genes=2000
    )

    adata_hvg = adata_harmony[:, adata_harmony.var["highly_variable"]].copy()

    sc.tl.pca(adata_hvg)
    ho = hm.run_harmony(
        adata_hvg.obsm["X_pca"],
        adata_hvg.obs,
        vars_use=["batch"]
    )

    adata_hvg.obsm["X_pca_harmony"] = np.array(ho.Z_corr)
    sc.pp.neighbors(adata_hvg, use_rep="X_pca_harmony")
    sc.tl.leiden(adata_hvg, resolution=0.5)

    X_harmony = adata_hvg.obsm["X_pca_harmony"]

    nn_harmony = pynndescent(
        X_harmony,
        n_neighbors=30,
        random_state=1000 + rep
    )

    ilisi = sm.ilisi_knn(
        nn_harmony,
        adata_hvg.obs["batch"].to_numpy(),
        scale=True
    )

    clisi = sm.clisi_knn(
        nn_harmony,
        adata_hvg.obs["cell_type"].to_numpy(),
        scale=True
    )

    ari = adjusted_rand_score(
        adata_hvg.obs["cell_type"],
        adata_hvg.obs["leiden"]
    )

    nmi = normalized_mutual_info_score(
        adata_hvg.obs["cell_type"],
        adata_hvg.obs["leiden"]
    )

    results_harmony_75.append({
        "method": "Harmony",
        "frac": 0.75,
        "repeat": rep,
        "n_cells": int(adata_hvg.n_obs),
        "n_hvg": int(adata_hvg.n_vars),
        "ilisi": float(ilisi),
        "clisi": float(clisi),
        "ari": float(ari),
        "nmi": float(nmi)
    })

    print(results_harmony_75[-1])
    del adata_sub, adata_harmony, adata_hvg, ho, nn_harmony, X_harmony
    gc.collect()

results_harmony_75_df = pd.DataFrame(results_harmony_75)

print("\nAll 3 repeats:")
print(results_harmony_75_df)

print("\nMean across 3 repeats:")
print(results_harmony_75_df[["ilisi", "clisi", "ari", "nmi"]].mean())

print("\nSD across 3 repeats:")
print(results_harmony_75_df[["ilisi", "clisi", "ari", "nmi"]].std())

/Users/meredithzhang/PHP2691hw/PythonProject5/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm



Running Harmony for 75% repeat 1...


2026-05-09 04:50:57,940 - harmonypy - INFO - Running Harmony
2026-05-09 04:50:57,941 - harmonypy - INFO -   Parameters:
2026-05-09 04:50:57,942 - harmonypy - INFO -     max_iter_harmony: 10
2026-05-09 04:50:57,942 - harmonypy - INFO -     max_iter_kmeans: 4
2026-05-09 04:50:57,942 - harmonypy - INFO -     epsilon_cluster: 0.001
2026-05-09 04:50:57,942 - harmonypy - INFO -     epsilon_harmony: 0.01
2026-05-09 04:50:57,942 - harmonypy - INFO -     nclust: 100
2026-05-09 04:50:57,942 - harmonypy - INFO -     block_size: 0.05
2026-05-09 04:50:57,942 - harmonypy - INFO -     lamb: dynamic (alpha=0.2)
2026-05-09 04:50:57,943 - harmonypy - INFO -     theta: [2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2.]
2026-05-09 04:50:57,943 - harmonypy - INFO -     sigma: [0.1 0.1 0.1 0.1 0.1]...
2026-05-09 04:50:57,943 - harmonypy - INFO -     verbose: True
2026-05-09 04:50:57,943 - harmonypy - INFO -     random_state: 0
2026-05-09 04:50:57,943 - harmonypy - INFO -   Data: 50 PCs × 18612 cells
2026-05-

{'method': 'Harmony', 'frac': 0.75, 'repeat': 1, 'n_cells': 18612, 'n_hvg': 2000, 'ilisi': 0.08218660205602646, 'clisi': 1.0, 'ari': 0.6323772522362721, 'nmi': 0.7751547454493039}

Running Harmony for 75% repeat 2...


2026-05-09 04:51:21,358 - harmonypy - INFO - Running Harmony
2026-05-09 04:51:21,358 - harmonypy - INFO -   Parameters:
2026-05-09 04:51:21,359 - harmonypy - INFO -     max_iter_harmony: 10
2026-05-09 04:51:21,359 - harmonypy - INFO -     max_iter_kmeans: 4
2026-05-09 04:51:21,359 - harmonypy - INFO -     epsilon_cluster: 0.001
2026-05-09 04:51:21,359 - harmonypy - INFO -     epsilon_harmony: 0.01
2026-05-09 04:51:21,359 - harmonypy - INFO -     nclust: 100
2026-05-09 04:51:21,360 - harmonypy - INFO -     block_size: 0.05
2026-05-09 04:51:21,360 - harmonypy - INFO -     lamb: dynamic (alpha=0.2)
2026-05-09 04:51:21,360 - harmonypy - INFO -     theta: [2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2.]
2026-05-09 04:51:21,360 - harmonypy - INFO -     sigma: [0.1 0.1 0.1 0.1 0.1]...
2026-05-09 04:51:21,360 - harmonypy - INFO -     verbose: True
2026-05-09 04:51:21,361 - harmonypy - INFO -     random_state: 0
2026-05-09 04:51:21,361 - harmonypy - INFO -   Data: 50 PCs × 18612 cells
2026-05-

{'method': 'Harmony', 'frac': 0.75, 'repeat': 2, 'n_cells': 18612, 'n_hvg': 2000, 'ilisi': 0.08184242248535156, 'clisi': 1.0, 'ari': 0.632027998577051, 'nmi': 0.7747462384348823}

Running Harmony for 75% repeat 3...


2026-05-09 04:51:33,696 - harmonypy - INFO - Running Harmony
2026-05-09 04:51:33,697 - harmonypy - INFO -   Parameters:
2026-05-09 04:51:33,697 - harmonypy - INFO -     max_iter_harmony: 10
2026-05-09 04:51:33,697 - harmonypy - INFO -     max_iter_kmeans: 4
2026-05-09 04:51:33,697 - harmonypy - INFO -     epsilon_cluster: 0.001
2026-05-09 04:51:33,697 - harmonypy - INFO -     epsilon_harmony: 0.01
2026-05-09 04:51:33,698 - harmonypy - INFO -     nclust: 100
2026-05-09 04:51:33,698 - harmonypy - INFO -     block_size: 0.05
2026-05-09 04:51:33,698 - harmonypy - INFO -     lamb: dynamic (alpha=0.2)
2026-05-09 04:51:33,698 - harmonypy - INFO -     theta: [2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2.]
2026-05-09 04:51:33,698 - harmonypy - INFO -     sigma: [0.1 0.1 0.1 0.1 0.1]...
2026-05-09 04:51:33,698 - harmonypy - INFO -     verbose: True
2026-05-09 04:51:33,699 - harmonypy - INFO -     random_state: 0
2026-05-09 04:51:33,699 - harmonypy - INFO -   Data: 50 PCs × 18612 cells
2026-05-

{'method': 'Harmony', 'frac': 0.75, 'repeat': 3, 'n_cells': 18612, 'n_hvg': 2000, 'ilisi': 0.08099675178527832, 'clisi': 1.0, 'ari': 0.6213433478102146, 'nmi': 0.7651146719520716}

All 3 repeats:
    method  frac  repeat  n_cells  n_hvg     ilisi  clisi       ari       nmi
0  Harmony  0.75       1    18612   2000  0.082187    1.0  0.632377  0.775155
1  Harmony  0.75       2    18612   2000  0.081842    1.0  0.632028  0.774746
2  Harmony  0.75       3    18612   2000  0.080997    1.0  0.621343  0.765115

Mean across 3 repeats:
ilisi    0.081675
clisi    1.000000
ari      0.628583
nmi      0.771672
dtype: float64

SD across 3 repeats:
ilisi    0.000612
clisi    0.000000
ari      0.006272
nmi      0.005682
dtype: float64


In [ ]:
import os
import gc
import numpy as np
import pandas as pd
import scanpy as sc
import scvi
import scib_metrics as sm
from scib_metrics.nearest_neighbors import pynndescent
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score

results_scvi_75 = []

for rep in [1, 2, 3]:
    file_path = os.path.expanduser(f"~/Desktop/adata_lung_75_rep{rep}.h5ad")
    adata_sub = sc.read_h5ad(file_path)

    scvi.settings.seed = 1000 + rep

    scvi.model.SCVI.setup_anndata(
        adata_sub,
        layer="counts",
        batch_key="batch"
    )

    model = scvi.model.SCVI(
        adata_sub,
        n_latent=30
    )

    model.train()

    adata_sub.obsm["X_scVI"] = model.get_latent_representation()

    sc.pp.neighbors(adata_sub, use_rep="X_scVI")
    sc.tl.leiden(adata_sub, resolution=0.5)

    X_scvi = adata_sub.obsm["X_scVI"]

    nn_scvi = pynndescent(
        X_scvi,
        n_neighbors=30,
        random_state=1000 + rep
    )

    ilisi = sm.ilisi_knn(
        nn_scvi,
        adata_sub.obs["batch"].to_numpy(),
        scale=True
    )

    clisi = sm.clisi_knn(
        nn_scvi,
        adata_sub.obs["cell_type"].to_numpy(),
        scale=True
    )

    ari = adjusted_rand_score(
        adata_sub.obs["cell_type"],
        adata_sub.obs["leiden"]
    )

    nmi = normalized_mutual_info_score(
        adata_sub.obs["cell_type"],
        adata_sub.obs["leiden"]
    )

    results_scvi_75.append({
        "method": "scVI",
        "frac": 0.75,
        "repeat": rep,
        "n_cells": int(adata_sub.n_obs),
        "n_genes": int(adata_sub.n_vars),
        "ilisi": float(ilisi),
        "clisi": float(clisi),
        "ari": float(ari),
        "nmi": float(nmi)
    })

    print(results_scvi_75[-1])

    del adata_sub, model, nn_scvi, X_scvi
    gc.collect()

results_scvi_75_df = pd.DataFrame(results_scvi_75)

print("\nAll 3 repeats:")
print(results_scvi_75_df)

print("\nMean across 3 repeats:")
print(results_scvi_75_df[["ilisi", "clisi", "ari", "nmi"]].mean())

print("\nSD across 3 repeats:")
print(results_scvi_75_df[["ilisi", "clisi", "ari", "nmi"]].std())


Running scVI for 75% repeat 1...


Seed set to 1001
/Users/meredithzhang/PHP2691hw/PythonProject5/.venv/lib/python3.11/site-packages/scvi/data/fields/_base_field.py:63: UserWarning: adata.layers[counts] does not contain unnormalized count data. Are you sure this is what you want?
  self.validate_field(adata)
/Users/meredithzhang/PHP2691hw/PythonProject5/.venv/lib/python3.11/site-packages/scvi/train/_trainrunner.py:98: UserWarning: `accelerator` has been automatically set to `cpu` although 'mps' exists. If you wish to run on mps backend, use explicitly accelerator='mps' in train function.In future releases it will become default for mps supported machines.
  accelerator, lightning_devices, device = parse_device_args(
GPU available: True (mps), used: False
TPU available: False, using: 0 TPU cores
/Users/meredithzhang/PHP2691hw/PythonProject5/.venv/lib/python3.11/site-packages/lightning/pytorch/trainer/setup.py:175: GPU available but not used. You can set it by doing `Trainer(accelerator='gpu')`.
💡 Tip: For seamless cloud 

Epoch 1/400:   0%|          | 0/400 [00:00<?, ?it/s]

/Users/meredithzhang/PHP2691hw/PythonProject5/.venv/lib/python3.11/site-packages/scvi/module/_vae.py:573: UserWarning: The value argument must be within the support of the distribution
  reconst_loss = -generative_outputs[MODULE_KEYS.PX_KEY].log_prob(x).sum(-1)
/Users/meredithzhang/PHP2691hw/PythonProject5/.venv/lib/python3.11/site-packages/scvi/module/_vae.py:573: UserWarning: The value argument must be within the support of the distribution
  reconst_loss = -generative_outputs[MODULE_KEYS.PX_KEY].log_prob(x).sum(-1)


Epoch 2/400:   0%|          | 1/400 [00:10<1:10:08, 10.55s/it, v_num=1, train_loss=5.92e+3]

...

Epoch 4/400:   1%|          | 3/400 [00:59<2:23:37, 21.71s/it, v_num=1, train_loss=5.25e+3]

/Users/meredithzhang/PHP2691hw/PythonProject5/.venv/lib/python3.11/site-packages/scvi/module/_vae.py:573: UserWarning: The value argument must be within the support of the distribution
  reconst_loss = -generative_outputs[MODULE_KEYS.PX_KEY].log_prob(x).sum(-1)
/Users/meredithzhang/PHP2691hw/PythonProject5/.venv/lib/python3.11/site-packages/scvi/module/_vae.py:573: UserWarning: The value argument must be within the support of the distribution
  reconst_loss = -generative_outputs[MODULE_KEYS.PX_KEY].log_prob(x).sum(-1)


Epoch 5/400:   1%|          | 4/400 [01:25<2:34:54, 23.47s/it, v_num=1, train_loss=5.19e+3]

/Users/meredithzhang/PHP2691hw/PythonProject5/.venv/lib/python3.11/site-packages/scvi/module/_vae.py:573: UserWarning: The value argument must be within the support of the distribution
  reconst_loss = -generative_outputs[MODULE_KEYS.PX_KEY].log_prob(x).sum(-1)
/Users/meredithzhang/PHP2691hw/PythonProject5/.venv/lib/python3.11/site-packages/scvi/module/_vae.py:573: UserWarning: The value argument must be within the support of the distribution
  reconst_loss = -generative_outputs[MODULE_KEYS.PX_KEY].log_prob(x).sum(-1)


Epoch 6/400:   1%|▏         | 5/400 [01:50<2:38:07, 24.02s/it, v_num=1, train_loss=5.14e+3]

In [1]:
import os
import numpy as np
import scanpy as sc
import scvi
import scib_metrics as sm
from scib_metrics.nearest_neighbors import pynndescent
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score

file_path = os.path.expanduser("~/Desktop/adata_lung_75_rep3.h5ad")
adata_sub = sc.read_h5ad(file_path)

print(adata_sub)

scvi.settings.seed = 1003

scvi.model.SCVI.setup_anndata(
    adata_sub,
    layer="counts",
    batch_key="batch"
)

model = scvi.model.SCVI(
    adata_sub,
    n_latent=30
)

model.train()

adata_sub.obsm["X_scVI"] = model.get_latent_representation()

sc.pp.neighbors(adata_sub, use_rep="X_scVI")
sc.tl.leiden(adata_sub, resolution=0.5)

X_scvi = adata_sub.obsm["X_scVI"]

nn_scvi = pynndescent(
    X_scvi,
    n_neighbors=30,
    random_state=1003
)

ilisi = sm.ilisi_knn(
    nn_scvi,
    adata_sub.obs["batch"].to_numpy(),
    scale=True
)

clisi = sm.clisi_knn(
    nn_scvi,
    adata_sub.obs["cell_type"].to_numpy(),
    scale=True
)

ari = adjusted_rand_score(
    adata_sub.obs["cell_type"],
    adata_sub.obs["leiden"]
)

nmi = normalized_mutual_info_score(
    adata_sub.obs["cell_type"],
    adata_sub.obs["leiden"]
)

result_rep3 = {
    "method": "scVI",
    "frac": 0.75,
    "repeat": 3,
    "n_cells": int(adata_sub.n_obs),
    "n_genes": int(adata_sub.n_vars),
    "ilisi": float(ilisi),
    "clisi": float(clisi),
    "ari": float(ari),
    "nmi": float(nmi)
}

print("\nscVI 75% repeat 3 result:")
print(result_rep3)

/Users/meredithzhang/PHP2691hw/PythonProject5/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Running scVI for 75% repeat 3...


Seed set to 1003
/Users/meredithzhang/PHP2691hw/PythonProject5/.venv/lib/python3.11/site-packages/scvi/data/fields/_base_field.py:63: UserWarning: adata.layers[counts] does not contain unnormalized count data. Are you sure this is what you want?
  self.validate_field(adata)
/Users/meredithzhang/PHP2691hw/PythonProject5/.venv/lib/python3.11/site-packages/scvi/train/_trainrunner.py:98: UserWarning: `accelerator` has been automatically set to `cpu` although 'mps' exists. If you wish to run on mps backend, use explicitly accelerator='mps' in train function.In future releases it will become default for mps supported machines.
  accelerator, lightning_devices, device = parse_device_args(
GPU available: True (mps), used: False
TPU available: False, using: 0 TPU cores
/Users/meredithzhang/PHP2691hw/PythonProject5/.venv/lib/python3.11/site-packages/lightning/pytorch/trainer/setup.py:175: GPU available but not used. You can set it by doing `Trainer(accelerator='gpu')`.
💡 Tip: For seamless cloud 

AnnData object with n_obs × n_vars = 18612 × 15139
    obs: 'dataset', 'location', 'nGene', 'nUMI', 'patientGroup', 'percent.mito', 'protocol', 'sanger_type', 'size_factors', 'sampling_method', 'batch', 'cell_type', 'donor'
    var: 'n_cells'
    layers: 'counts'
Epoch 1/400:   0%|          | 0/400 [00:00<?, ?it/s]

/Users/meredithzhang/PHP2691hw/PythonProject5/.venv/lib/python3.11/site-packages/scvi/module/_vae.py:573: UserWarning: The value argument must be within the support of the distribution
  reconst_loss = -generative_outputs[MODULE_KEYS.PX_KEY].log_prob(x).sum(-1)
/Users/meredithzhang/PHP2691hw/PythonProject5/.venv/lib/python3.11/site-packages/scvi/module/_vae.py:573: UserWarning: The value argument must be within the support of the distribution
  reconst_loss = -generative_outputs[MODULE_KEYS.PX_KEY].log_prob(x).sum(-1)


...

Epoch 400/400: 100%|██████████| 400/400 [1:32:10<00:00, 12.63s/it, v_num=1, train_loss=4.58e+3]

`Trainer.fit` stopped: `max_epochs=400` reached.


Epoch 400/400: 100%|██████████| 400/400 [1:32:10<00:00, 13.83s/it, v_num=1, train_loss=4.58e+3]


/var/folders/3w/yklb0sxs0ss499k2l0cb08z80000gn/T/ipykernel_82072/1850840706.py:40: FutureWarning: In the future, the default backend for leiden will be igraph instead of leidenalg.

 To achieve the future defaults please pass: flavor="igraph" and n_iterations=2.  directed must also be False to work with igraph's implementation.
  sc.tl.leiden(adata_sub, resolution=0.5)



scVI 75% repeat 3 result:
{'method': 'scVI', 'frac': 0.75, 'repeat': 3, 'n_cells': 18612, 'n_genes': 15139, 'ilisi': 0.06978019326925278, 'clisi': 0.998734176158905, 'ari': 0.6037430233170478, 'nmi': 0.7716766888004006}


In [3]:
import os
import pandas as pd
import scib_metrics as sm
from scib_metrics.nearest_neighbors import pynndescent
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score

def compute_metrics_from_one_file(
    csv_path,
    batch_col="batch",
    celltype_col="cell_type",
    cluster_col="cluster"
):
    df = pd.read_csv(csv_path)

    embed_cols = [c for c in df.columns if c.startswith("PC_")]
    X = df[embed_cols].to_numpy()

    nn = pynndescent(
        X,
        n_neighbors=30,
        random_state=0
    )

    ilisi = sm.ilisi_knn(
        nn,
        df[batch_col].to_numpy(),
        scale=True
    )

    clisi = sm.clisi_knn(
        nn,
        df[celltype_col].to_numpy(),
        scale=True
    )

    ari = adjusted_rand_score(
        df[celltype_col],
        df[cluster_col]
    )

    nmi = normalized_mutual_info_score(
        df[celltype_col],
        df[cluster_col]
    )

    return {
        "n_cells": len(df),
        "ilisi": float(ilisi),
        "clisi": float(clisi),
        "ari": float(ari),
        "nmi": float(nmi)
    }

results_seurat_75 = []
base_dir = os.path.expanduser("~/Desktop")

for rep in [1, 2, 3]:
    file_path = os.path.join(base_dir, f"lung_seurat_75_rep{rep}.csv")

    metrics = compute_metrics_from_one_file(
        csv_path=file_path,
        batch_col="batch",
        celltype_col="cell_type",
        cluster_col="cluster"
    )

    results_seurat_75.append({
        "method": "Seurat",
        "frac": 0.75,
        "repeat": rep,
        **metrics
    })

results_seurat_75_df = pd.DataFrame(results_seurat_75)

print("Repeat-level results:")
print(results_seurat_75_df)

print("\nMean:")
print(results_seurat_75_df[["ilisi", "clisi", "ari", "nmi"]].mean())

print("\nSD:")
print(results_seurat_75_df[["ilisi", "clisi", "ari", "nmi"]].std())

Repeat-level results:
   method  frac  repeat  n_cells     ilisi     clisi       ari       nmi
0  Seurat  0.75       1    18612  0.103405  0.998607  0.558552  0.716957
1  Seurat  0.75       2    18612  0.112511  0.998074  0.537863  0.707341
2  Seurat  0.75       3    18612  0.104144  0.998269  0.556083  0.718861

Mean:
ilisi    0.106687
clisi    0.998317
ari      0.550833
nmi      0.714387
dtype: float64

SD:
ilisi    0.005057
clisi    0.000270
ari      0.011300
nmi      0.006176
dtype: float64


In [5]:
import os
import gc
import numpy as np
import pandas as pd
import scanpy as sc
import scib_metrics as sm
from scib_metrics.nearest_neighbors import pynndescent
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score

results_fastmnn_75 = []

for rep in [1, 2, 3]:
    file_path = os.path.expanduser(f"~/Desktop/lung75_{rep}_fastmnn.h5ad")
    adata_mnn = sc.read_h5ad(file_path)

    # Use corrected embedding from fastMNN
    X_mnn = adata_mnn.obsm["corrected"]

    # Neighbors / Leiden
    sc.pp.neighbors(adata_mnn, use_rep="corrected")
    sc.tl.leiden(adata_mnn, resolution=0.5)

    # LISI metrics
    nn_mnn = pynndescent(
        X_mnn,
        n_neighbors=30,
        random_state=1000 + rep
    )

    ilisi = sm.ilisi_knn(
        nn_mnn,
        adata_mnn.obs["batch"].to_numpy(),
        scale=True
    )

    clisi = sm.clisi_knn(
        nn_mnn,
        adata_mnn.obs["cell_type"].to_numpy(),
        scale=True
    )

    # ARI / NMI
    ari = adjusted_rand_score(
        adata_mnn.obs["cell_type"],
        adata_mnn.obs["leiden"]
    )

    nmi = normalized_mutual_info_score(
        adata_mnn.obs["cell_type"],
        adata_mnn.obs["leiden"]
    )

    results_fastmnn_75.append({
        "method": "fastMNN",
        "frac": 0.75,
        "repeat": rep,
        "n_cells": int(adata_mnn.n_obs),
        "n_genes": int(adata_mnn.n_vars),
        "ilisi": float(ilisi),
        "clisi": float(clisi),
        "ari": float(ari),
        "nmi": float(nmi)
    })

    print(results_fastmnn_75[-1])

    del adata_mnn, X_mnn, nn_mnn
    gc.collect()

results_fastmnn_75_df = pd.DataFrame(results_fastmnn_75)

print("\nRepeat-level results:")
print(results_fastmnn_75_df)

print("\nMean:")
print(results_fastmnn_75_df[["ilisi", "clisi", "ari", "nmi"]].mean())

print("\nSD:")
print(results_fastmnn_75_df[["ilisi", "clisi", "ari", "nmi"]].std())

{'method': 'fastMNN', 'frac': 0.75, 'repeat': 1, 'n_cells': 18612, 'n_genes': 2000, 'ilisi': 0.06829682737588882, 'clisi': 1.0, 'ari': 0.6236515131201714, 'nmi': 0.7741492066945402}
{'method': 'fastMNN', 'frac': 0.75, 'repeat': 2, 'n_cells': 18612, 'n_genes': 2000, 'ilisi': 0.06768868863582611, 'clisi': 1.0, 'ari': 0.6327696009916653, 'nmi': 0.777860975538818}
{'method': 'fastMNN', 'frac': 0.75, 'repeat': 3, 'n_cells': 18612, 'n_genes': 2000, 'ilisi': 0.06757138669490814, 'clisi': 1.0, 'ari': 0.6321282680749319, 'nmi': 0.7735175736257832}

Repeat-level results:
    method  frac  repeat  n_cells  n_genes     ilisi  clisi       ari  \
0  fastMNN  0.75       1    18612     2000  0.068297    1.0  0.623652   
1  fastMNN  0.75       2    18612     2000  0.067689    1.0  0.632770   
2  fastMNN  0.75       3    18612     2000  0.067571    1.0  0.632128   

        nmi  
0  0.774149  
1  0.777861  
2  0.773518  

Mean:
ilisi    0.067852
clisi    1.000000
ari      0.629516
nmi      0.775176
dtyp

In [6]:
file_path = os.path.expanduser("~/Desktop/adata_lung_qc.h5ad")
adata_lung_qc = sc.read_h5ad(file_path)

def stratified_subsample_adata(adata, group_key="batch", frac=0.50, random_state=0):
    rng = np.random.default_rng(random_state)
    selected_idx = []

    obs = adata.obs.copy()

    for group, idx in obs.groupby(group_key).indices.items():
        idx = np.array(list(idx))
        n_group = len(idx)
        n_take = max(1, int(np.floor(n_group * frac)))
        chosen = rng.choice(idx, size=n_take, replace=False)
        selected_idx.extend(chosen.tolist())

    selected_idx = np.array(selected_idx)
    return adata[selected_idx].copy()

adata_lung_50_rep1 = stratified_subsample_adata(
    adata_lung_qc,
    group_key="batch",
    frac=0.50,
    random_state=1001
)

adata_lung_50_rep2 = stratified_subsample_adata(
    adata_lung_qc,
    group_key="batch",
    frac=0.50,
    random_state=1002
)

adata_lung_50_rep3 = stratified_subsample_adata(
    adata_lung_qc,
    group_key="batch",
    frac=0.50,
    random_state=1003
)

adata_lung_50_rep1.write(os.path.expanduser("~/Desktop/adata_lung_50_rep1.h5ad"))
adata_lung_50_rep2.write(os.path.expanduser("~/Desktop/adata_lung_50_rep2.h5ad"))
adata_lung_50_rep3.write(os.path.expanduser("~/Desktop/adata_lung_50_rep3.h5ad"))

/var/folders/3w/yklb0sxs0ss499k2l0cb08z80000gn/T/ipykernel_82072/2405683355.py:10: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  for group, idx in obs.groupby(group_key).indices.items():


In [7]:
import os
import gc
import numpy as np
import pandas as pd
import scanpy as sc
import harmonypy as hm
import scib_metrics as sm
from scib_metrics.nearest_neighbors import pynndescent
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score

results_harmony_50 = []

for rep in [1, 2, 3]:
    file_path = os.path.expanduser(f"~/Desktop/adata_lung_50_rep{rep}.h5ad")
    adata_sub = sc.read_h5ad(file_path)

    adata_harmony = adata_sub.copy()
    adata_harmony.X = adata_harmony.layers["counts"].copy()

    sc.pp.normalize_total(adata_harmony, target_sum=1e4)
    sc.pp.log1p(adata_harmony)

    sc.pp.highly_variable_genes(
        adata_harmony,
        flavor="seurat",
        batch_key="batch",
        n_top_genes=2000
    )

    adata_hvg = adata_harmony[:, adata_harmony.var["highly_variable"]].copy()

    sc.tl.pca(adata_hvg)

    ho = hm.run_harmony(
        adata_hvg.obsm["X_pca"],
        adata_hvg.obs,
        vars_use=["batch"]
    )

    adata_hvg.obsm["X_pca_harmony"] = np.array(ho.Z_corr)

    sc.pp.neighbors(adata_hvg, use_rep="X_pca_harmony")
    sc.tl.leiden(adata_hvg, resolution=0.5)

    X_harmony = adata_hvg.obsm["X_pca_harmony"]

    nn_harmony = pynndescent(
        X_harmony,
        n_neighbors=30,
        random_state=1000 + rep
    )

    ilisi = sm.ilisi_knn(
        nn_harmony,
        adata_hvg.obs["batch"].to_numpy(),
        scale=True
    )

    clisi = sm.clisi_knn(
        nn_harmony,
        adata_hvg.obs["cell_type"].to_numpy(),
        scale=True
    )

    ari = adjusted_rand_score(
        adata_hvg.obs["cell_type"],
        adata_hvg.obs["leiden"]
    )

    nmi = normalized_mutual_info_score(
        adata_hvg.obs["cell_type"],
        adata_hvg.obs["leiden"]
    )

    results_harmony_50.append({
        "method": "Harmony",
        "frac": 0.50,
        "repeat": rep,
        "n_cells": int(adata_hvg.n_obs),
        "n_hvg": int(adata_hvg.n_vars),
        "ilisi": float(ilisi),
        "clisi": float(clisi),
        "ari": float(ari),
        "nmi": float(nmi)
    })

    print(results_harmony_50[-1])

    del adata_sub, adata_harmony, adata_hvg, ho, nn_harmony, X_harmony
    gc.collect()

results_harmony_50_df = pd.DataFrame(results_harmony_50)

print(results_harmony_50_df)

print(results_harmony_50_df[["ilisi", "clisi", "ari", "nmi"]].mean())

print(results_harmony_50_df[["ilisi", "clisi", "ari", "nmi"]].std())

2026-05-10 03:18:22,064 - harmonypy - INFO - Running Harmony
2026-05-10 03:18:22,065 - harmonypy - INFO -   Parameters:
2026-05-10 03:18:22,065 - harmonypy - INFO -     max_iter_harmony: 10
2026-05-10 03:18:22,065 - harmonypy - INFO -     max_iter_kmeans: 4
2026-05-10 03:18:22,065 - harmonypy - INFO -     epsilon_cluster: 0.001
2026-05-10 03:18:22,066 - harmonypy - INFO -     epsilon_harmony: 0.01
2026-05-10 03:18:22,066 - harmonypy - INFO -     nclust: 100
2026-05-10 03:18:22,066 - harmonypy - INFO -     block_size: 0.05
2026-05-10 03:18:22,066 - harmonypy - INFO -     lamb: dynamic (alpha=0.2)
2026-05-10 03:18:22,066 - harmonypy - INFO -     theta: [2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2.]
2026-05-10 03:18:22,067 - harmonypy - INFO -     sigma: [0.1 0.1 0.1 0.1 0.1]...
2026-05-10 03:18:22,067 - harmonypy - INFO -     verbose: True
2026-05-10 03:18:22,067 - harmonypy - INFO -     random_state: 0
2026-05-10 03:18:22,067 - harmonypy - INFO -   Data: 50 PCs × 12408 cells
2026-05-

{'method': 'Harmony', 'frac': 0.5, 'repeat': 1, 'n_cells': 12408, 'n_hvg': 2000, 'ilisi': 0.08691685646772385, 'clisi': 0.9999855160713196, 'ari': 0.6259333051918674, 'nmi': 0.7755719972105511}


2026-05-10 03:18:30,806 - harmonypy - INFO - Running Harmony
2026-05-10 03:18:30,807 - harmonypy - INFO -   Parameters:
2026-05-10 03:18:30,807 - harmonypy - INFO -     max_iter_harmony: 10
2026-05-10 03:18:30,807 - harmonypy - INFO -     max_iter_kmeans: 4
2026-05-10 03:18:30,808 - harmonypy - INFO -     epsilon_cluster: 0.001
2026-05-10 03:18:30,808 - harmonypy - INFO -     epsilon_harmony: 0.01
2026-05-10 03:18:30,808 - harmonypy - INFO -     nclust: 100
2026-05-10 03:18:30,808 - harmonypy - INFO -     block_size: 0.05
2026-05-10 03:18:30,809 - harmonypy - INFO -     lamb: dynamic (alpha=0.2)
2026-05-10 03:18:30,809 - harmonypy - INFO -     theta: [2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2.]
2026-05-10 03:18:30,809 - harmonypy - INFO -     sigma: [0.1 0.1 0.1 0.1 0.1]...
2026-05-10 03:18:30,809 - harmonypy - INFO -     verbose: True
2026-05-10 03:18:30,810 - harmonypy - INFO -     random_state: 0
2026-05-10 03:18:30,810 - harmonypy - INFO -   Data: 50 PCs × 12408 cells
2026-05-

{'method': 'Harmony', 'frac': 0.5, 'repeat': 2, 'n_cells': 12408, 'n_hvg': 2000, 'ilisi': 0.0879182368516922, 'clisi': 1.0, 'ari': 0.6045543804287733, 'nmi': 0.7656638736563613}


2026-05-10 03:18:38,496 - harmonypy - INFO - Running Harmony
2026-05-10 03:18:38,498 - harmonypy - INFO -   Parameters:
2026-05-10 03:18:38,498 - harmonypy - INFO -     max_iter_harmony: 10
2026-05-10 03:18:38,498 - harmonypy - INFO -     max_iter_kmeans: 4
2026-05-10 03:18:38,498 - harmonypy - INFO -     epsilon_cluster: 0.001
2026-05-10 03:18:38,499 - harmonypy - INFO -     epsilon_harmony: 0.01
2026-05-10 03:18:38,499 - harmonypy - INFO -     nclust: 100
2026-05-10 03:18:38,499 - harmonypy - INFO -     block_size: 0.05
2026-05-10 03:18:38,499 - harmonypy - INFO -     lamb: dynamic (alpha=0.2)
2026-05-10 03:18:38,499 - harmonypy - INFO -     theta: [2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2.]
2026-05-10 03:18:38,499 - harmonypy - INFO -     sigma: [0.1 0.1 0.1 0.1 0.1]...
2026-05-10 03:18:38,500 - harmonypy - INFO -     verbose: True
2026-05-10 03:18:38,500 - harmonypy - INFO -     random_state: 0
2026-05-10 03:18:38,500 - harmonypy - INFO -   Data: 50 PCs × 12408 cells
2026-05-

{'method': 'Harmony', 'frac': 0.5, 'repeat': 3, 'n_cells': 12408, 'n_hvg': 2000, 'ilisi': 0.08636455237865448, 'clisi': 1.0, 'ari': 0.6470552779286228, 'nmi': 0.779970096858657}
    method  frac  repeat  n_cells  n_hvg     ilisi     clisi       ari  \
0  Harmony   0.5       1    12408   2000  0.086917  0.999986  0.625933   
1  Harmony   0.5       2    12408   2000  0.087918  1.000000  0.604554   
2  Harmony   0.5       3    12408   2000  0.086365  1.000000  0.647055   

        nmi  
0  0.775572  
1  0.765664  
2  0.779970  
ilisi    0.087067
clisi    0.999995
ari      0.625848
nmi      0.773735
dtype: float64
ilisi    0.000788
clisi    0.000008
ari      0.021251
nmi      0.007328
dtype: float64


In [8]:
def compute_metrics_from_one_file(
    csv_path,
    batch_col="batch",
    celltype_col="cell_type",
    cluster_col="cluster"
):
    df = pd.read_csv(csv_path)

    embed_cols = [c for c in df.columns if c.startswith("PC_")]
    X = df[embed_cols].to_numpy()

    nn = pynndescent(
        X,
        n_neighbors=30,
        random_state=0
    )

    ilisi = sm.ilisi_knn(
        nn,
        df[batch_col].to_numpy(),
        scale=True
    )

    clisi = sm.clisi_knn(
        nn,
        df[celltype_col].to_numpy(),
        scale=True
    )

    ari = adjusted_rand_score(
        df[celltype_col],
        df[cluster_col]
    )

    nmi = normalized_mutual_info_score(
        df[celltype_col],
        df[cluster_col]
    )

    return {
        "n_cells": len(df),
        "ilisi": float(ilisi),
        "clisi": float(clisi),
        "ari": float(ari),
        "nmi": float(nmi)
    }

results_seurat_50 = []
base_dir = os.path.expanduser("~/Desktop")

for rep in [1, 2, 3]:
    file_path = os.path.join(base_dir, f"lung_seurat_50_rep{rep}.csv")

    metrics = compute_metrics_from_one_file(
        csv_path=file_path,
        batch_col="batch",
        celltype_col="cell_type",
        cluster_col="cluster"
    )

    results_seurat_50.append({
        "method": "Seurat",
        "frac": 0.50,
        "repeat": rep,
        **metrics
    })

results_seurat_50_df = pd.DataFrame(results_seurat_50)

print("Repeat-level results:")
print(results_seurat_50_df)

print("\nMean:")
print(results_seurat_50_df[["ilisi", "clisi", "ari", "nmi"]].mean())

print("\nSD:")
print(results_seurat_50_df[["ilisi", "clisi", "ari", "nmi"]].std())

Repeat-level results:
   method  frac  repeat  n_cells     ilisi     clisi       ari       nmi
0  Seurat   0.5       1    12408  0.109519  0.998684  0.543085  0.712258
1  Seurat   0.5       2    12408  0.106508  0.998067  0.523731  0.706646
2  Seurat   0.5       3    12408  0.107031  0.999057  0.520777  0.709401

Mean:
ilisi    0.107686
clisi    0.998603
ari      0.529198
nmi      0.709435
dtype: float64

SD:
ilisi    0.001609
clisi    0.000500
ari      0.012117
nmi      0.002806
dtype: float64


In [9]:
results_fastmnn_50 = []

for rep in [1, 2, 3]:
    file_path = os.path.expanduser(f"~/Desktop/lung50_{rep}_fastmnn.h5ad")
    adata_mnn = sc.read_h5ad(file_path)

    X_mnn = adata_mnn.obsm["corrected"]

    sc.pp.neighbors(adata_mnn, use_rep="corrected")
    sc.tl.leiden(adata_mnn, resolution=0.5)

    nn_mnn = pynndescent(
        X_mnn,
        n_neighbors=30,
        random_state=1000 + rep
    )

    ilisi = sm.ilisi_knn(
        nn_mnn,
        adata_mnn.obs["batch"].to_numpy(),
        scale=True
    )

    clisi = sm.clisi_knn(
        nn_mnn,
        adata_mnn.obs["cell_type"].to_numpy(),
        scale=True
    )

    ari = adjusted_rand_score(
        adata_mnn.obs["cell_type"],
        adata_mnn.obs["leiden"]
    )

    nmi = normalized_mutual_info_score(
        adata_mnn.obs["cell_type"],
        adata_mnn.obs["leiden"]
    )

    results_fastmnn_50.append({
        "method": "fastMNN",
        "frac": 0.50,
        "repeat": rep,
        "n_cells": int(adata_mnn.n_obs),
        "n_genes": int(adata_mnn.n_vars),
        "ilisi": float(ilisi),
        "clisi": float(clisi),
        "ari": float(ari),
        "nmi": float(nmi)
    })

    print(results_fastmnn_50[-1])

    del adata_mnn, X_mnn, nn_mnn
    gc.collect()

results_fastmnn_50_df = pd.DataFrame(results_fastmnn_50)

print("\nRepeat-level results:")
print(results_fastmnn_50_df)

print("\nMean:")
print(results_fastmnn_50_df[["ilisi", "clisi", "ari", "nmi"]].mean())

print("\nSD:")
print(results_fastmnn_50_df[["ilisi", "clisi", "ari", "nmi"]].std())

{'method': 'fastMNN', 'frac': 0.5, 'repeat': 1, 'n_cells': 12408, 'n_genes': 2000, 'ilisi': 0.07257852703332901, 'clisi': 1.0, 'ari': 0.6030276423733822, 'nmi': 0.750746172796957}
{'method': 'fastMNN', 'frac': 0.5, 'repeat': 2, 'n_cells': 12408, 'n_genes': 2000, 'ilisi': 0.07142918556928635, 'clisi': 1.0, 'ari': 0.618902728890952, 'nmi': 0.7686745907148652}
{'method': 'fastMNN', 'frac': 0.5, 'repeat': 3, 'n_cells': 12408, 'n_genes': 2000, 'ilisi': 0.07198844105005264, 'clisi': 0.999896228313446, 'ari': 0.6391475090606262, 'nmi': 0.7688464197124932}

Repeat-level results:
    method  frac  repeat  n_cells  n_genes     ilisi     clisi       ari  \
0  fastMNN   0.5       1    12408     2000  0.072579  1.000000  0.603028   
1  fastMNN   0.5       2    12408     2000  0.071429  1.000000  0.618903   
2  fastMNN   0.5       3    12408     2000  0.071988  0.999896  0.639148   

        nmi  
0  0.750746  
1  0.768675  
2  0.768846  

Mean:
ilisi    0.071999
clisi    0.999965
ari      0.620359


In [16]:
file_path = os.path.expanduser("~/Desktop/adata_lung_qc.h5ad")
adata_lung_qc = sc.read_h5ad(file_path)

def stratified_subsample_adata(adata, group_key="batch", frac=0.25, random_state=0):
    rng = np.random.default_rng(random_state)
    selected_idx = []

    obs = adata.obs.copy()

    for group, idx in obs.groupby(group_key).indices.items():
        idx = np.array(list(idx))
        n_group = len(idx)
        n_take = max(1, int(np.floor(n_group * frac)))
        chosen = rng.choice(idx, size=n_take, replace=False)
        selected_idx.extend(chosen.tolist())

    selected_idx = np.array(selected_idx)
    return adata[selected_idx].copy()

adata_lung_25_rep1 = stratified_subsample_adata(
    adata_lung_qc,
    group_key="batch",
    frac=0.25,
    random_state=1001
)

adata_lung_25_rep2 = stratified_subsample_adata(
    adata_lung_qc,
    group_key="batch",
    frac=0.25,
    random_state=1002
)

adata_lung_25_rep3 = stratified_subsample_adata(
    adata_lung_qc,
    group_key="batch",
    frac=0.25,
    random_state=1003
)

adata_lung_25_rep1.write(os.path.expanduser("~/Desktop/adata_lung_25_rep1.h5ad"))
adata_lung_25_rep2.write(os.path.expanduser("~/Desktop/adata_lung_25_rep2.h5ad"))
adata_lung_25_rep3.write(os.path.expanduser("~/Desktop/adata_lung_25_rep3.h5ad"))

/var/folders/3w/yklb0sxs0ss499k2l0cb08z80000gn/T/ipykernel_82072/3764823123.py:10: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  for group, idx in obs.groupby(group_key).indices.items():


In [17]:
results_harmony_25 = []

for rep in [1, 2, 3]:
    file_path = os.path.expanduser(f"~/Desktop/adata_lung_25_rep{rep}.h5ad")
    adata_sub = sc.read_h5ad(file_path)

    adata_harmony = adata_sub.copy()
    adata_harmony.X = adata_harmony.layers["counts"].copy()

    sc.pp.normalize_total(adata_harmony, target_sum=1e4)
    sc.pp.log1p(adata_harmony)

    sc.pp.highly_variable_genes(
        adata_harmony,
        flavor="seurat",
        batch_key="batch",
        n_top_genes=2000
    )

    adata_hvg = adata_harmony[:, adata_harmony.var["highly_variable"]].copy()

    sc.tl.pca(adata_hvg)

    ho = hm.run_harmony(
        adata_hvg.obsm["X_pca"],
        adata_hvg.obs,
        vars_use=["batch"]
    )

    adata_hvg.obsm["X_pca_harmony"] = np.array(ho.Z_corr)

    sc.pp.neighbors(adata_hvg, use_rep="X_pca_harmony")
    sc.tl.leiden(adata_hvg, resolution=0.5)

    X_harmony = adata_hvg.obsm["X_pca_harmony"]

    nn_harmony = pynndescent(
        X_harmony,
        n_neighbors=30,
        random_state=1000 + rep
    )

    ilisi = sm.ilisi_knn(
        nn_harmony,
        adata_hvg.obs["batch"].to_numpy(),
        scale=True
    )

    clisi = sm.clisi_knn(
        nn_harmony,
        adata_hvg.obs["cell_type"].to_numpy(),
        scale=True
    )

    ari = adjusted_rand_score(
        adata_hvg.obs["cell_type"],
        adata_hvg.obs["leiden"]
    )

    nmi = normalized_mutual_info_score(
        adata_hvg.obs["cell_type"],
        adata_hvg.obs["leiden"]
    )

    results_harmony_25.append({
        "method": "Harmony",
        "frac": 0.25,
        "repeat": rep,
        "n_cells": int(adata_hvg.n_obs),
        "n_hvg": int(adata_hvg.n_vars),
        "ilisi": float(ilisi),
        "clisi": float(clisi),
        "ari": float(ari),
        "nmi": float(nmi)
    })

    print(results_harmony_25[-1])

    del adata_sub, adata_harmony, adata_hvg, ho, nn_harmony, X_harmony
    gc.collect()

results_harmony_25_df = pd.DataFrame(results_harmony_25)

print(results_harmony_25_df)

print(results_harmony_25_df[["ilisi", "clisi", "ari", "nmi"]].mean())

print(results_harmony_25_df[["ilisi", "clisi", "ari", "nmi"]].std())

2026-05-10 11:41:55,298 - harmonypy - INFO - Running Harmony
2026-05-10 11:41:55,299 - harmonypy - INFO -   Parameters:
2026-05-10 11:41:55,299 - harmonypy - INFO -     max_iter_harmony: 10
2026-05-10 11:41:55,299 - harmonypy - INFO -     max_iter_kmeans: 4
2026-05-10 11:41:55,300 - harmonypy - INFO -     epsilon_cluster: 0.001
2026-05-10 11:41:55,300 - harmonypy - INFO -     epsilon_harmony: 0.01
2026-05-10 11:41:55,300 - harmonypy - INFO -     nclust: 100
2026-05-10 11:41:55,300 - harmonypy - INFO -     block_size: 0.05
2026-05-10 11:41:55,300 - harmonypy - INFO -     lamb: dynamic (alpha=0.2)
2026-05-10 11:41:55,300 - harmonypy - INFO -     theta: [2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2.]
2026-05-10 11:41:55,301 - harmonypy - INFO -     sigma: [0.1 0.1 0.1 0.1 0.1]...
2026-05-10 11:41:55,301 - harmonypy - INFO -     verbose: True
2026-05-10 11:41:55,301 - harmonypy - INFO -     random_state: 0
2026-05-10 11:41:55,301 - harmonypy - INFO -   Data: 50 PCs × 6200 cells
2026-05-1

{'method': 'Harmony', 'frac': 0.25, 'repeat': 1, 'n_cells': 6200, 'n_hvg': 2000, 'ilisi': 0.09786300361156464, 'clisi': 0.9995470643043518, 'ari': 0.5921785720013414, 'nmi': 0.768395357430345}


2026-05-10 11:41:58,787 - harmonypy - INFO - Running Harmony
2026-05-10 11:41:58,787 - harmonypy - INFO -   Parameters:
2026-05-10 11:41:58,788 - harmonypy - INFO -     max_iter_harmony: 10
2026-05-10 11:41:58,788 - harmonypy - INFO -     max_iter_kmeans: 4
2026-05-10 11:41:58,788 - harmonypy - INFO -     epsilon_cluster: 0.001
2026-05-10 11:41:58,789 - harmonypy - INFO -     epsilon_harmony: 0.01
2026-05-10 11:41:58,789 - harmonypy - INFO -     nclust: 100
2026-05-10 11:41:58,789 - harmonypy - INFO -     block_size: 0.05
2026-05-10 11:41:58,789 - harmonypy - INFO -     lamb: dynamic (alpha=0.2)
2026-05-10 11:41:58,790 - harmonypy - INFO -     theta: [2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2.]
2026-05-10 11:41:58,790 - harmonypy - INFO -     sigma: [0.1 0.1 0.1 0.1 0.1]...
2026-05-10 11:41:58,790 - harmonypy - INFO -     verbose: True
2026-05-10 11:41:58,790 - harmonypy - INFO -     random_state: 0
2026-05-10 11:41:58,790 - harmonypy - INFO -   Data: 50 PCs × 6200 cells
2026-05-1

{'method': 'Harmony', 'frac': 0.25, 'repeat': 2, 'n_cells': 6200, 'n_hvg': 2000, 'ilisi': 0.09494663774967194, 'clisi': 0.9991052150726318, 'ari': 0.6155468276391476, 'nmi': 0.7756933957992932}


2026-05-10 11:42:01,814 - harmonypy - INFO - Running Harmony
2026-05-10 11:42:01,814 - harmonypy - INFO -   Parameters:
2026-05-10 11:42:01,814 - harmonypy - INFO -     max_iter_harmony: 10
2026-05-10 11:42:01,814 - harmonypy - INFO -     max_iter_kmeans: 4
2026-05-10 11:42:01,815 - harmonypy - INFO -     epsilon_cluster: 0.001
2026-05-10 11:42:01,815 - harmonypy - INFO -     epsilon_harmony: 0.01
2026-05-10 11:42:01,815 - harmonypy - INFO -     nclust: 100
2026-05-10 11:42:01,815 - harmonypy - INFO -     block_size: 0.05
2026-05-10 11:42:01,815 - harmonypy - INFO -     lamb: dynamic (alpha=0.2)
2026-05-10 11:42:01,815 - harmonypy - INFO -     theta: [2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2.]
2026-05-10 11:42:01,816 - harmonypy - INFO -     sigma: [0.1 0.1 0.1 0.1 0.1]...
2026-05-10 11:42:01,816 - harmonypy - INFO -     verbose: True
2026-05-10 11:42:01,816 - harmonypy - INFO -     random_state: 0
2026-05-10 11:42:01,816 - harmonypy - INFO -   Data: 50 PCs × 6200 cells
2026-05-1

{'method': 'Harmony', 'frac': 0.25, 'repeat': 3, 'n_cells': 6200, 'n_hvg': 2000, 'ilisi': 0.09678968042135239, 'clisi': 0.9994414448738098, 'ari': 0.6281611620828176, 'nmi': 0.774397227692623}
    method  frac  repeat  n_cells  n_hvg     ilisi     clisi       ari  \
0  Harmony  0.25       1     6200   2000  0.097863  0.999547  0.592179   
1  Harmony  0.25       2     6200   2000  0.094947  0.999105  0.615547   
2  Harmony  0.25       3     6200   2000  0.096790  0.999441  0.628161   

        nmi  
0  0.768395  
1  0.775693  
2  0.774397  
ilisi    0.096533
clisi    0.999365
ari      0.611962
nmi      0.772829
dtype: float64
ilisi    0.001475
clisi    0.000231
ari      0.018257
nmi      0.003894
dtype: float64


In [18]:
def compute_metrics_from_one_file(
    csv_path,
    batch_col="batch",
    celltype_col="cell_type",
    cluster_col="cluster"
):
    df = pd.read_csv(csv_path)

    embed_cols = [c for c in df.columns if c.startswith("PC_")]
    X = df[embed_cols].to_numpy()

    nn = pynndescent(
        X,
        n_neighbors=30,
        random_state=0
    )

    ilisi = sm.ilisi_knn(
        nn,
        df[batch_col].to_numpy(),
        scale=True
    )

    clisi = sm.clisi_knn(
        nn,
        df[celltype_col].to_numpy(),
        scale=True
    )

    ari = adjusted_rand_score(
        df[celltype_col],
        df[cluster_col]
    )

    nmi = normalized_mutual_info_score(
        df[celltype_col],
        df[cluster_col]
    )

    return {
        "n_cells": len(df),
        "ilisi": float(ilisi),
        "clisi": float(clisi),
        "ari": float(ari),
        "nmi": float(nmi)
    }

results_seurat_25 = []
base_dir = os.path.expanduser("~/Desktop")

for rep in [1, 2, 3]:
    file_path = os.path.join(base_dir, f"lung_seurat_25_rep{rep}.csv")

    metrics = compute_metrics_from_one_file(
        csv_path=file_path,
        batch_col="batch",
        celltype_col="cell_type",
        cluster_col="cluster"
    )

    results_seurat_25.append({
        "method": "Seurat",
        "frac": 0.25,
        "repeat": rep,
        **metrics
    })

results_seurat_25_df = pd.DataFrame(results_seurat_25)

print("Repeat-level results:")
print(results_seurat_25_df)

print("\nMean:")
print(results_seurat_25_df[["ilisi", "clisi", "ari", "nmi"]].mean())

print("\nSD:")
print(results_seurat_25_df[["ilisi", "clisi", "ari", "nmi"]].std())

Repeat-level results:
   method  frac  repeat  n_cells     ilisi     clisi       ari       nmi
0  Seurat  0.25       1     6200  0.116848  0.995985  0.546708  0.702394
1  Seurat  0.25       2     6200  0.121616  0.995397  0.556531  0.702300
2  Seurat  0.25       3     6200  0.122102  0.995634  0.509245  0.695368

Mean:
ilisi    0.120189
clisi    0.995672
ari      0.537495
nmi      0.700021
dtype: float64

SD:
ilisi    0.002903
clisi    0.000296
ari      0.024953
nmi      0.004030
dtype: float64


In [1]:
import os
import gc
import numpy as np
import pandas as pd
import scanpy as sc
import scib_metrics as sm
from scib_metrics.nearest_neighbors import pynndescent
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score

results_fastmnn_25 = []

for rep in [1, 2, 3]:
    file_path = os.path.expanduser(f"~/Desktop/lung25_{rep}_fastmnn.h5ad")
    adata_mnn = sc.read_h5ad(file_path)

    X_mnn = adata_mnn.obsm["corrected"]

    sc.pp.neighbors(adata_mnn, use_rep="corrected")
    sc.tl.leiden(adata_mnn, resolution=0.5)

    nn_mnn = pynndescent(
        X_mnn,
        n_neighbors=30,
        random_state=1000 + rep
    )

    ilisi = sm.ilisi_knn(
        nn_mnn,
        adata_mnn.obs["batch"].to_numpy(),
        scale=True
    )

    clisi = sm.clisi_knn(
        nn_mnn,
        adata_mnn.obs["cell_type"].to_numpy(),
        scale=True
    )

    ari = adjusted_rand_score(
        adata_mnn.obs["cell_type"],
        adata_mnn.obs["leiden"]
    )

    nmi = normalized_mutual_info_score(
        adata_mnn.obs["cell_type"],
        adata_mnn.obs["leiden"]
    )

    results_fastmnn_25.append({
        "method": "fastMNN",
        "frac": 0.25,
        "repeat": rep,
        "n_cells": int(adata_mnn.n_obs),
        "n_genes": int(adata_mnn.n_vars),
        "ilisi": float(ilisi),
        "clisi": float(clisi),
        "ari": float(ari),
        "nmi": float(nmi)
    })

    print(results_fastmnn_25[-1])

    del adata_mnn, X_mnn, nn_mnn
    gc.collect()

results_fastmnn_25_df = pd.DataFrame(results_fastmnn_25)

print("\nRepeat-level results:")
print(results_fastmnn_25_df)

print("\nMean:")
print(results_fastmnn_25_df[["ilisi", "clisi", "ari", "nmi"]].mean())

print("\nSD:")
print(results_fastmnn_25_df[["ilisi", "clisi", "ari", "nmi"]].std())

/Users/meredithzhang/PHP2691hw/PythonProject5/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/var/folders/3w/yklb0sxs0ss499k2l0cb08z80000gn/T/ipykernel_96564/2451797982.py:19: FutureWarning: In the future, the default backend for leiden will be igraph instead of leidenalg.

 To achieve the future defaults please pass: flavor="igraph" and n_iterations=2.  directed must also be False to work with igraph's implementation.
  sc.tl.leiden(adata_mnn, resolution=0.5)


{'method': 'fastMNN', 'frac': 0.25, 'repeat': 1, 'n_cells': 6200, 'n_genes': 2000, 'ilisi': 0.078342504799366, 'clisi': 0.9986321926116943, 'ari': 0.6184862538973349, 'nmi': 0.7702154120088042}
{'method': 'fastMNN', 'frac': 0.25, 'repeat': 2, 'n_cells': 6200, 'n_genes': 2000, 'ilisi': 0.07875251770019531, 'clisi': 0.9986335635185242, 'ari': 0.6146469585377994, 'nmi': 0.7633900475007793}
{'method': 'fastMNN', 'frac': 0.25, 'repeat': 3, 'n_cells': 6200, 'n_genes': 2000, 'ilisi': 0.07776905596256256, 'clisi': 0.9990920424461365, 'ari': 0.6638132429076499, 'nmi': 0.7793267770909642}

Repeat-level results:
    method  frac  repeat  n_cells  n_genes     ilisi     clisi       ari  \
0  fastMNN  0.25       1     6200     2000  0.078343  0.998632  0.618486   
1  fastMNN  0.25       2     6200     2000  0.078753  0.998634  0.614647   
2  fastMNN  0.25       3     6200     2000  0.077769  0.999092  0.663813   

        nmi  
0  0.770215  
1  0.763390  
2  0.779327  

Mean:
ilisi    0.078288
clisi

In [ ]:
import os
import gc
import numpy as np
import pandas as pd
import scanpy as sc
import scvi
import scib_metrics as sm
from scib_metrics.nearest_neighbors import pynndescent
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score

results_scvi_50 = []

for rep in [1, 2, 3]:

    file_path = os.path.expanduser(f"~/Desktop/adata_lung_50_rep{rep}.h5ad")
    adata_sub = sc.read_h5ad(file_path)

    scvi.settings.seed = 1000 + rep

    scvi.model.SCVI.setup_anndata(
        adata_sub,
        layer="counts",
        batch_key="batch"
    )

    model = scvi.model.SCVI(
        adata_sub,
        n_latent=30
    )

    model.train()

    adata_sub.obsm["X_scVI"] = model.get_latent_representation()

    sc.pp.neighbors(adata_sub, use_rep="X_scVI")
    sc.tl.leiden(adata_sub, resolution=0.5)

    X_scvi = adata_sub.obsm["X_scVI"]

    nn_scvi = pynndescent(
        X_scvi,
        n_neighbors=30,
        random_state=1000 + rep
    )

    ilisi = sm.ilisi_knn(
        nn_scvi,
        adata_sub.obs["batch"].to_numpy(),
        scale=True
    )

    clisi = sm.clisi_knn(
        nn_scvi,
        adata_sub.obs["cell_type"].to_numpy(),
        scale=True
    )

    ari = adjusted_rand_score(
        adata_sub.obs["cell_type"],
        adata_sub.obs["leiden"]
    )

    nmi = normalized_mutual_info_score(
        adata_sub.obs["cell_type"],
        adata_sub.obs["leiden"]
    )

    results_scvi_50.append({
        "method": "scVI",
        "frac": 0.50,
        "repeat": rep,
        "n_cells": int(adata_sub.n_obs),
        "n_genes": int(adata_sub.n_vars),
        "ilisi": float(ilisi),
        "clisi": float(clisi),
        "ari": float(ari),
        "nmi": float(nmi)
    })

    print(results_scvi_50[-1])

    del adata_sub, model, nn_scvi, X_scvi
    gc.collect()

results_scvi_50_df = pd.DataFrame(results_scvi_50)

print("\nAll 3 repeats:")
print(results_scvi_50_df)

print("\nMean across 3 repeats:")
print(results_scvi_50_df[["ilisi", "clisi", "ari", "nmi"]].mean())

print("\nSD across 3 repeats:")
print(results_scvi_50_df[["ilisi", "clisi", "ari", "nmi"]].std())


Running scVI for 50% repeat 1...


Seed set to 1001
/Users/meredithzhang/PHP2691hw/PythonProject5/.venv/lib/python3.11/site-packages/scvi/data/fields/_base_field.py:63: UserWarning: adata.layers[counts] does not contain unnormalized count data. Are you sure this is what you want?
  self.validate_field(adata)
/Users/meredithzhang/PHP2691hw/PythonProject5/.venv/lib/python3.11/site-packages/scvi/train/_trainrunner.py:98: UserWarning: `accelerator` has been automatically set to `cpu` although 'mps' exists. If you wish to run on mps backend, use explicitly accelerator='mps' in train function.In future releases it will become default for mps supported machines.
  accelerator, lightning_devices, device = parse_device_args(
GPU available: True (mps), used: False
TPU available: False, using: 0 TPU cores
/Users/meredithzhang/PHP2691hw/PythonProject5/.venv/lib/python3.11/site-packages/lightning/pytorch/trainer/setup.py:175: GPU available but not used. You can set it by doing `Trainer(accelerator='gpu')`.
💡 Tip: For seamless cloud 

Epoch 1/400:   0%|          | 0/400 [00:00<?, ?it/s]

/Users/meredithzhang/PHP2691hw/PythonProject5/.venv/lib/python3.11/site-packages/scvi/module/_vae.py:573: UserWarning: The value argument must be within the support of the distribution
  reconst_loss = -generative_outputs[MODULE_KEYS.PX_KEY].log_prob(x).sum(-1)
/Users/meredithzhang/PHP2691hw/PythonProject5/.venv/lib/python3.11/site-packages/scvi/module/_vae.py:573: UserWarning: The value argument must be within the support of the distribution
  reconst_loss = -generative_outputs[MODULE_KEYS.PX_KEY].log_prob(x).sum(-1)


Epoch 2/400:   0%|          | 1/400 [00:09<1:00:58,  9.17s/it, v_num=1, train_loss=6.15e+3]

...

/Users/meredithzhang/PHP2691hw/PythonProject5/.venv/lib/python3.11/site-packages/scvi/module/_vae.py:573: UserWarning: The value argument must be within the support of the distribution
  reconst_loss = -generative_outputs[MODULE_KEYS.PX_KEY].log_prob(x).sum(-1)
/Users/meredithzhang/PHP2691hw/PythonProject5/.venv/lib/python3.11/site-packages/scvi/module/_vae.py:573: UserWarning: The value argument must be within the support of the distribution
  reconst_loss = -generative_outputs[MODULE_KEYS.PX_KEY].log_prob(x).sum(-1)


Epoch 54/400:  13%|█▎        | 53/400 [08:17<1:00:32, 10.47s/it, v_num=1, train_loss=4.66e+3]

/Users/meredithzhang/PHP2691hw/PythonProject5/.venv/lib/python3.11/site-packages/scvi/module/_vae.py:573: UserWarning: The value argument must be within the support of the distribution
  reconst_loss = -generative_outputs[MODULE_KEYS.PX_KEY].log_prob(x).sum(-1)
/Users/meredithzhang/PHP2691hw/PythonProject5/.venv/lib/python3.11/site-packages/scvi/module/_vae.py:573: UserWarning: The value argument must be within the support of the distribution
  reconst_loss = -generative_outputs[MODULE_KEYS.PX_KEY].log_prob(x).sum(-1)


Epoch 55/400:  14%|█▎        | 54/400 [08:26<58:35, 10.16s/it, v_num=1, train_loss=4.66e+3]  

/Users/meredithzhang/PHP2691hw/PythonProject5/.venv/lib/python3.11/site-packages/scvi/module/_vae.py:573: UserWarning: The value argument must be within the support of the distribution
  reconst_loss = -generative_outputs[MODULE_KEYS.PX_KEY].log_prob(x).sum(-1)
/Users/meredithzhang/PHP2691hw/PythonProject5/.venv/lib/python3.11/site-packages/scvi/module/_vae.py:573: UserWarning: The value argument must be within the support of the distribution
  reconst_loss = -generative_outputs[MODULE_KEYS.PX_KEY].log_prob(x).sum(-1)


In [1]:
import os
import numpy as np
import scanpy as sc
import scvi
import scib_metrics as sm
from scib_metrics.nearest_neighbors import pynndescent
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score

file_path = os.path.expanduser("~/Desktop/adata_lung_50_rep3.h5ad")
adata_sub = sc.read_h5ad(file_path)

print(adata_sub)

scvi.settings.seed = 1003

scvi.model.SCVI.setup_anndata(
    adata_sub,
    layer="counts",
    batch_key="batch"
)

model = scvi.model.SCVI(
    adata_sub,
    n_latent=30
)

model.train()

adata_sub.obsm["X_scVI"] = model.get_latent_representation()

sc.pp.neighbors(adata_sub, use_rep="X_scVI")
sc.tl.leiden(adata_sub, resolution=0.5)

X_scvi = adata_sub.obsm["X_scVI"]

nn_scvi = pynndescent(
    X_scvi,
    n_neighbors=30,
    random_state=1003
)

ilisi = sm.ilisi_knn(
    nn_scvi,
    adata_sub.obs["batch"].to_numpy(),
    scale=True
)

clisi = sm.clisi_knn(
    nn_scvi,
    adata_sub.obs["cell_type"].to_numpy(),
    scale=True
)

ari = adjusted_rand_score(
    adata_sub.obs["cell_type"],
    adata_sub.obs["leiden"]
)

nmi = normalized_mutual_info_score(
    adata_sub.obs["cell_type"],
    adata_sub.obs["leiden"]
)

result_rep3 = {
    "method": "scVI",
    "frac": 0.50,
    "repeat": 3,
    "n_cells": int(adata_sub.n_obs),
    "n_genes": int(adata_sub.n_vars),
    "ilisi": float(ilisi),
    "clisi": float(clisi),
    "ari": float(ari),
    "nmi": float(nmi)
}

print("\nscVI 50% repeat 3 result:")
print(result_rep3)

/Users/meredithzhang/PHP2691hw/PythonProject5/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Running scVI for 50% repeat 3...


Seed set to 1003
/Users/meredithzhang/PHP2691hw/PythonProject5/.venv/lib/python3.11/site-packages/scvi/data/fields/_base_field.py:63: UserWarning: adata.layers[counts] does not contain unnormalized count data. Are you sure this is what you want?
  self.validate_field(adata)
/Users/meredithzhang/PHP2691hw/PythonProject5/.venv/lib/python3.11/site-packages/scvi/train/_trainrunner.py:98: UserWarning: `accelerator` has been automatically set to `cpu` although 'mps' exists. If you wish to run on mps backend, use explicitly accelerator='mps' in train function.In future releases it will become default for mps supported machines.
  accelerator, lightning_devices, device = parse_device_args(
GPU available: True (mps), used: False
TPU available: False, using: 0 TPU cores
/Users/meredithzhang/PHP2691hw/PythonProject5/.venv/lib/python3.11/site-packages/lightning/pytorch/trainer/setup.py:175: GPU available but not used. You can set it by doing `Trainer(accelerator='gpu')`.
💡 Tip: For seamless cloud 

AnnData object with n_obs × n_vars = 12408 × 15139
    obs: 'dataset', 'location', 'nGene', 'nUMI', 'patientGroup', 'percent.mito', 'protocol', 'sanger_type', 'size_factors', 'sampling_method', 'batch', 'cell_type', 'donor'
    var: 'n_cells'
    layers: 'counts'
Epoch 1/400:   0%|          | 0/400 [00:00<?, ?it/s]

/Users/meredithzhang/PHP2691hw/PythonProject5/.venv/lib/python3.11/site-packages/scvi/module/_vae.py:573: UserWarning: The value argument must be within the support of the distribution
  reconst_loss = -generative_outputs[MODULE_KEYS.PX_KEY].log_prob(x).sum(-1)
/Users/meredithzhang/PHP2691hw/PythonProject5/.venv/lib/python3.11/site-packages/scvi/module/_vae.py:573: UserWarning: The value argument must be within the support of the distribution
  reconst_loss = -generative_outputs[MODULE_KEYS.PX_KEY].log_prob(x).sum(-1)


...

Epoch 400/400: 100%|██████████| 400/400 [1:09:03<00:00,  8.07s/it, v_num=1, train_loss=4.57e+3]

`Trainer.fit` stopped: `max_epochs=400` reached.


Epoch 400/400: 100%|██████████| 400/400 [1:09:03<00:00, 10.36s/it, v_num=1, train_loss=4.57e+3]


/var/folders/3w/yklb0sxs0ss499k2l0cb08z80000gn/T/ipykernel_1514/298335676.py:34: FutureWarning: In the future, the default backend for leiden will be igraph instead of leidenalg.

 To achieve the future defaults please pass: flavor="igraph" and n_iterations=2.  directed must also be False to work with igraph's implementation.
  sc.tl.leiden(adata_sub, resolution=0.5)



scVI 50% repeat 3 result:
{'method': 'scVI', 'frac': 0.5, 'repeat': 3, 'n_cells': 12408, 'n_genes': 15139, 'ilisi': 0.07490846514701843, 'clisi': 0.9979391694068909, 'ari': 0.5854659222104517, 'nmi': 0.7654199289451528}


In [ ]:
import os
import gc
import numpy as np
import pandas as pd
import scanpy as sc
import scvi
import scib_metrics as sm
from scib_metrics.nearest_neighbors import pynndescent
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score

results_scvi_25 = []

for rep in [1, 2, 3]:

    file_path = os.path.expanduser(f"~/Desktop/adata_lung_25_rep{rep}.h5ad")
    adata_sub = sc.read_h5ad(file_path)

    scvi.settings.seed = 1000 + rep

    scvi.model.SCVI.setup_anndata(
        adata_sub,
        layer="counts",
        batch_key="batch"
    )

    model = scvi.model.SCVI(
        adata_sub,
        n_latent=30
    )

    model.train()

    adata_sub.obsm["X_scVI"] = model.get_latent_representation()

    sc.pp.neighbors(adata_sub, use_rep="X_scVI")
    sc.tl.leiden(adata_sub, resolution=0.5)

    X_scvi = adata_sub.obsm["X_scVI"]

    nn_scvi = pynndescent(
        X_scvi,
        n_neighbors=30,
        random_state=1000 + rep
    )

    ilisi = sm.ilisi_knn(
        nn_scvi,
        adata_sub.obs["batch"].to_numpy(),
        scale=True
    )

    clisi = sm.clisi_knn(
        nn_scvi,
        adata_sub.obs["cell_type"].to_numpy(),
        scale=True
    )

    ari = adjusted_rand_score(
        adata_sub.obs["cell_type"],
        adata_sub.obs["leiden"]
    )

    nmi = normalized_mutual_info_score(
        adata_sub.obs["cell_type"],
        adata_sub.obs["leiden"]
    )

    results_scvi_25.append({
        "method": "scVI",
        "frac": 0.25,
        "repeat": rep,
        "n_cells": int(adata_sub.n_obs),
        "n_genes": int(adata_sub.n_vars),
        "ilisi": float(ilisi),
        "clisi": float(clisi),
        "ari": float(ari),
        "nmi": float(nmi)
    })

    print(results_scvi_25[-1])

    del adata_sub, model, nn_scvi, X_scvi
    gc.collect()

results_scvi_25_df = pd.DataFrame(results_scvi_25)

print("\nAll 3 repeats:")
print(results_scvi_25_df)

print("\nMean across 3 repeats:")
print(results_scvi_25_df[["ilisi", "clisi", "ari", "nmi"]].mean())

print("\nSD across 3 repeats:")
print(results_scvi_25_df[["ilisi", "clisi", "ari", "nmi"]].std())

Seed set to 1001



Running scVI for 25% repeat 1...


/Users/meredithzhang/PHP2691hw/PythonProject5/.venv/lib/python3.11/site-packages/scvi/data/fields/_base_field.py:63: UserWarning: adata.layers[counts] does not contain unnormalized count data. Are you sure this is what you want?
  self.validate_field(adata)
/Users/meredithzhang/PHP2691hw/PythonProject5/.venv/lib/python3.11/site-packages/scvi/train/_trainrunner.py:98: UserWarning: `accelerator` has been automatically set to `cpu` although 'mps' exists. If you wish to run on mps backend, use explicitly accelerator='mps' in train function.In future releases it will become default for mps supported machines.
  accelerator, lightning_devices, device = parse_device_args(
GPU available: True (mps), used: False
TPU available: False, using: 0 TPU cores
/Users/meredithzhang/PHP2691hw/PythonProject5/.venv/lib/python3.11/site-packages/lightning/pytorch/trainer/setup.py:175: GPU available but not used. You can set it by doing `Trainer(accelerator='gpu')`.
💡 Tip: For seamless cloud logging and exper

Epoch 1/400:   0%|          | 0/400 [00:00<?, ?it/s]

/Users/meredithzhang/PHP2691hw/PythonProject5/.venv/lib/python3.11/site-packages/scvi/module/_vae.py:573: UserWarning: The value argument must be within the support of the distribution
  reconst_loss = -generative_outputs[MODULE_KEYS.PX_KEY].log_prob(x).sum(-1)
/Users/meredithzhang/PHP2691hw/PythonProject5/.venv/lib/python3.11/site-packages/scvi/module/_vae.py:573: UserWarning: The value argument must be within the support of the distribution
  reconst_loss = -generative_outputs[MODULE_KEYS.PX_KEY].log_prob(x).sum(-1)


...

/Users/meredithzhang/PHP2691hw/PythonProject5/.venv/lib/python3.11/site-packages/scvi/module/_vae.py:573: UserWarning: The value argument must be within the support of the distribution
  reconst_loss = -generative_outputs[MODULE_KEYS.PX_KEY].log_prob(x).sum(-1)
/Users/meredithzhang/PHP2691hw/PythonProject5/.venv/lib/python3.11/site-packages/scvi/module/_vae.py:573: UserWarning: The value argument must be within the support of the distribution
  reconst_loss = -generative_outputs[MODULE_KEYS.PX_KEY].log_prob(x).sum(-1)


Epoch 20/400:   5%|▍         | 19/400 [01:06<22:22,  3.52s/it, v_num=1, train_loss=5.06e+3]

/Users/meredithzhang/PHP2691hw/PythonProject5/.venv/lib/python3.11/site-packages/scvi/module/_vae.py:573: UserWarning: The value argument must be within the support of the distribution
  reconst_loss = -generative_outputs[MODULE_KEYS.PX_KEY].log_prob(x).sum(-1)
/Users/meredithzhang/PHP2691hw/PythonProject5/.venv/lib/python3.11/site-packages/scvi/module/_vae.py:573: UserWarning: The value argument must be within the support of the distribution
  reconst_loss = -generative_outputs[MODULE_KEYS.PX_KEY].log_prob(x).sum(-1)


Epoch 21/400:   5%|▌         | 20/400 [01:09<21:58,  3.47s/it, v_num=1, train_loss=5.04e+3]

/Users/meredithzhang/PHP2691hw/PythonProject5/.venv/lib/python3.11/site-packages/scvi/module/_vae.py:573: UserWarning: The value argument must be within the support of the distribution
  reconst_loss = -generative_outputs[MODULE_KEYS.PX_KEY].log_prob(x).sum(-1)
/Users/meredithzhang/PHP2691hw/PythonProject5/.venv/lib/python3.11/site-packages/scvi/module/_vae.py:573: UserWarning: The value argument must be within the support of the distribution
  reconst_loss = -generative_outputs[MODULE_KEYS.PX_KEY].log_prob(x).sum(-1)


In [1]:
import os
import numpy as np
import scanpy as sc
import scvi
import scib_metrics as sm
from scib_metrics.nearest_neighbors import pynndescent
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score

file_path = os.path.expanduser("~/Desktop/adata_lung_25_rep2.h5ad")
adata_sub = sc.read_h5ad(file_path)

print(adata_sub)

scvi.settings.seed = 1002

scvi.model.SCVI.setup_anndata(
    adata_sub,
    layer="counts",
    batch_key="batch"
)

model = scvi.model.SCVI(
    adata_sub,
    n_latent=30
)

model.train()

adata_sub.obsm["X_scVI"] = model.get_latent_representation()

sc.pp.neighbors(adata_sub, use_rep="X_scVI")
sc.tl.leiden(adata_sub, resolution=0.5)

X_scvi = adata_sub.obsm["X_scVI"]

nn_scvi = pynndescent(
    X_scvi,
    n_neighbors=30,
    random_state=1002
)

ilisi = sm.ilisi_knn(
    nn_scvi,
    adata_sub.obs["batch"].to_numpy(),
    scale=True
)

clisi = sm.clisi_knn(
    nn_scvi,
    adata_sub.obs["cell_type"].to_numpy(),
    scale=True
)

ari = adjusted_rand_score(
    adata_sub.obs["cell_type"],
    adata_sub.obs["leiden"]
)

nmi = normalized_mutual_info_score(
    adata_sub.obs["cell_type"],
    adata_sub.obs["leiden"]
)

result_rep2 = {
    "method": "scVI",
    "frac": 0.25,
    "repeat": 2,
    "n_cells": int(adata_sub.n_obs),
    "n_genes": int(adata_sub.n_vars),
    "ilisi": float(ilisi),
    "clisi": float(clisi),
    "ari": float(ari),
    "nmi": float(nmi)
}

print("\nscVI 25% repeat 2 result:")
print(result_rep2)

/Users/meredithzhang/PHP2691hw/PythonProject5/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Seed set to 1002


Running scVI for 25% repeat 2...
AnnData object with n_obs × n_vars = 6200 × 15139
    obs: 'dataset', 'location', 'nGene', 'nUMI', 'patientGroup', 'percent.mito', 'protocol', 'sanger_type', 'size_factors', 'sampling_method', 'batch', 'cell_type', 'donor'
    var: 'n_cells'
    layers: 'counts'


/Users/meredithzhang/PHP2691hw/PythonProject5/.venv/lib/python3.11/site-packages/scvi/data/fields/_base_field.py:63: UserWarning: adata.layers[counts] does not contain unnormalized count data. Are you sure this is what you want?
  self.validate_field(adata)
/Users/meredithzhang/PHP2691hw/PythonProject5/.venv/lib/python3.11/site-packages/scvi/train/_trainrunner.py:98: UserWarning: `accelerator` has been automatically set to `cpu` although 'mps' exists. If you wish to run on mps backend, use explicitly accelerator='mps' in train function.In future releases it will become default for mps supported machines.
  accelerator, lightning_devices, device = parse_device_args(
GPU available: True (mps), used: False
TPU available: False, using: 0 TPU cores
/Users/meredithzhang/PHP2691hw/PythonProject5/.venv/lib/python3.11/site-packages/lightning/pytorch/trainer/setup.py:175: GPU available but not used. You can set it by doing `Trainer(accelerator='gpu')`.
💡 Tip: For seamless cloud logging and exper

Epoch 1/400:   0%|          | 0/400 [00:00<?, ?it/s]

/Users/meredithzhang/PHP2691hw/PythonProject5/.venv/lib/python3.11/site-packages/scvi/module/_vae.py:573: UserWarning: The value argument must be within the support of the distribution
  reconst_loss = -generative_outputs[MODULE_KEYS.PX_KEY].log_prob(x).sum(-1)
/Users/meredithzhang/PHP2691hw/PythonProject5/.venv/lib/python3.11/site-packages/scvi/module/_vae.py:573: UserWarning: The value argument must be within the support of the distribution
  reconst_loss = -generative_outputs[MODULE_KEYS.PX_KEY].log_prob(x).sum(-1)


...

Epoch 400/400: 100%|██████████| 400/400 [33:54<00:00,  3.52s/it, v_num=1, train_loss=4.53e+3]

`Trainer.fit` stopped: `max_epochs=400` reached.


Epoch 400/400: 100%|██████████| 400/400 [33:54<00:00,  5.09s/it, v_num=1, train_loss=4.53e+3]


/var/folders/3w/yklb0sxs0ss499k2l0cb08z80000gn/T/ipykernel_3955/1904098128.py:34: FutureWarning: In the future, the default backend for leiden will be igraph instead of leidenalg.

 To achieve the future defaults please pass: flavor="igraph" and n_iterations=2.  directed must also be False to work with igraph's implementation.
  sc.tl.leiden(adata_sub, resolution=0.5)



scVI 25% repeat 2 result:
{'method': 'scVI', 'frac': 0.25, 'repeat': 2, 'n_cells': 6200, 'n_genes': 15139, 'ilisi': 0.08849800378084183, 'clisi': 0.9958866834640503, 'ari': 0.5849985624290244, 'nmi': 0.7643696961573947}


In [4]:
import os
import numpy as np
import scanpy as sc
import scvi
import scib_metrics as sm
from scib_metrics.nearest_neighbors import pynndescent
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score

file_path = os.path.expanduser("~/Desktop/adata_lung_25_rep3.h5ad")
adata_sub = sc.read_h5ad(file_path)

print(adata_sub)

scvi.settings.seed = 1003

scvi.model.SCVI.setup_anndata(
    adata_sub,
    layer="counts",
    batch_key="batch"
)

model = scvi.model.SCVI(
    adata_sub,
    n_latent=30
)

model.train()

adata_sub.obsm["X_scVI"] = model.get_latent_representation()

sc.pp.neighbors(adata_sub, use_rep="X_scVI")
sc.tl.leiden(adata_sub, resolution=0.5)

X_scvi = adata_sub.obsm["X_scVI"]

nn_scvi = pynndescent(
    X_scvi,
    n_neighbors=30,
    random_state=1003
)

ilisi = sm.ilisi_knn(
    nn_scvi,
    adata_sub.obs["batch"].to_numpy(),
    scale=True
)

clisi = sm.clisi_knn(
    nn_scvi,
    adata_sub.obs["cell_type"].to_numpy(),
    scale=True
)

ari = adjusted_rand_score(
    adata_sub.obs["cell_type"],
    adata_sub.obs["leiden"]
)

nmi = normalized_mutual_info_score(
    adata_sub.obs["cell_type"],
    adata_sub.obs["leiden"]
)

result_rep3 = {
    "method": "scVI",
    "frac": 0.25,
    "repeat": 3,
    "n_cells": int(adata_sub.n_obs),
    "n_genes": int(adata_sub.n_vars),
    "ilisi": float(ilisi),
    "clisi": float(clisi),
    "ari": float(ari),
    "nmi": float(nmi)
}

print("\nscVI 25% repeat 3 result:")
print(result_rep3)

Seed set to 1003


Running scVI for 25% repeat 3...
AnnData object with n_obs × n_vars = 6200 × 15139
    obs: 'dataset', 'location', 'nGene', 'nUMI', 'patientGroup', 'percent.mito', 'protocol', 'sanger_type', 'size_factors', 'sampling_method', 'batch', 'cell_type', 'donor'
    var: 'n_cells'
    layers: 'counts'


/Users/meredithzhang/PHP2691hw/PythonProject5/.venv/lib/python3.11/site-packages/scvi/data/fields/_base_field.py:63: UserWarning: adata.layers[counts] does not contain unnormalized count data. Are you sure this is what you want?
  self.validate_field(adata)
/Users/meredithzhang/PHP2691hw/PythonProject5/.venv/lib/python3.11/site-packages/scvi/train/_trainrunner.py:98: UserWarning: `accelerator` has been automatically set to `cpu` although 'mps' exists. If you wish to run on mps backend, use explicitly accelerator='mps' in train function.In future releases it will become default for mps supported machines.
  accelerator, lightning_devices, device = parse_device_args(
GPU available: True (mps), used: False
TPU available: False, using: 0 TPU cores
/Users/meredithzhang/PHP2691hw/PythonProject5/.venv/lib/python3.11/site-packages/lightning/pytorch/trainer/setup.py:175: GPU available but not used. You can set it by doing `Trainer(accelerator='gpu')`.
💡 Tip: For seamless cloud logging and exper

Epoch 1/400:   0%|          | 0/400 [00:00<?, ?it/s]

/Users/meredithzhang/PHP2691hw/PythonProject5/.venv/lib/python3.11/site-packages/scvi/module/_vae.py:573: UserWarning: The value argument must be within the support of the distribution
  reconst_loss = -generative_outputs[MODULE_KEYS.PX_KEY].log_prob(x).sum(-1)
/Users/meredithzhang/PHP2691hw/PythonProject5/.venv/lib/python3.11/site-packages/scvi/module/_vae.py:573: UserWarning: The value argument must be within the support of the distribution
  reconst_loss = -generative_outputs[MODULE_KEYS.PX_KEY].log_prob(x).sum(-1)


...

Epoch 400/400: 100%|█████████▉| 399/400 [23:17<00:03,  3.55s/it, v_num=1, train_loss=4.51e+3]

/Users/meredithzhang/PHP2691hw/PythonProject5/.venv/lib/python3.11/site-packages/scvi/module/_vae.py:573: UserWarning: The value argument must be within the support of the distribution
  reconst_loss = -generative_outputs[MODULE_KEYS.PX_KEY].log_prob(x).sum(-1)
/Users/meredithzhang/PHP2691hw/PythonProject5/.venv/lib/python3.11/site-packages/scvi/module/_vae.py:573: UserWarning: The value argument must be within the support of the distribution
  reconst_loss = -generative_outputs[MODULE_KEYS.PX_KEY].log_prob(x).sum(-1)


Epoch 400/400: 100%|██████████| 400/400 [23:21<00:00,  3.56s/it, v_num=1, train_loss=4.51e+3]

`Trainer.fit` stopped: `max_epochs=400` reached.


Epoch 400/400: 100%|██████████| 400/400 [23:21<00:00,  3.50s/it, v_num=1, train_loss=4.51e+3]

scVI 25% repeat 3 result:
{'method': 'scVI', 'frac': 0.25, 'repeat': 3, 'n_cells': 6200, 'n_genes': 15139, 'ilisi': 0.0894673690199852, 'clisi': 0.9965178966522217, 'ari': 0.5808421592016624, 'nmi': 0.7698367141984165}
